# Task 1 — Wheat kernels: source download and benchmark preparation

## Goal

Download the official University of Copenhagen archive and recreate the benchmark directly from `NITSingleSeed.mat`. The source `Calibration` partition becomes the benchmark training set and the source `Validation` partition becomes the benchmark test set.

The original download is held in a temporary directory and removed at the end. Only the generated benchmark CSV files remain in the task directory.

## Original source

- Dataset page: <https://ucphchemometrics.com/datasets/>
- Archive: <https://sid.erda.dk/share_redirect/dLQ6VHNshw/Wheat%20kernels%20.zip>

## Setup

Set the task parameters, locate the output directory, and define download and verification helpers. The comparison runs before the generated files replace any existing local copies.

In [1]:
TASK_FOLDER = "1-Wheat_kernels_protein"
INCLUDE_INDEX = True
COMPARISON_ATOL = 1e-12
SOURCE_URL = "https://sid.erda.dk/share_redirect/dLQ6VHNshw/Wheat%20kernels%20.zip"

In [2]:
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import io
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
from scipy.io import loadmat


def find_task_dir(folder_name):
    """Support running from either the task directory or repository root."""
    candidates = [Path.cwd(), Path.cwd() / "datasets" / folder_name]
    for candidate in candidates:
        if candidate.name == folder_name and (candidate / "download_data.md").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Run this notebook from {folder_name!r} or from the repository root."
    )


def download(url, destination):
    """Download a source file without retaining it in the repository."""
    request = Request(url, headers={"User-Agent": "NIRBENCH-DL/0.1 dataset preparation"})
    with urlopen(request, timeout=120) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"Downloaded {destination.name} ({destination.stat().st_size:,} bytes)")


def csv_roundtrip(frame, include_index):
    """Represent a generated frame as it will appear after CSV serialization."""
    buffer = io.StringIO()
    frame.to_csv(buffer, index=include_index, lineterminator="\r\n")
    buffer.seek(0)
    return pd.read_csv(buffer)


def verify_and_write(train_frame, test_frame, include_index, atol):
    """Compare with published partitions when present, then write both CSV files."""
    results = []
    for split, generated in (("train", train_frame), ("test", test_frame)):
        output_path = TASK_DIR / f"data_{split}.csv"
        serialized = csv_roundtrip(generated, include_index)

        if output_path.exists():
            published = pd.read_csv(output_path)
            pd.testing.assert_frame_equal(
                serialized,
                published,
                check_dtype=False,
                check_exact=False,
                rtol=0,
                atol=atol,
            )
            max_delta = np.max(
                np.abs(
                    serialized.select_dtypes(include="number").to_numpy(dtype=float)
                    - published.select_dtypes(include="number").to_numpy(dtype=float)
                )
            )
            print(
                f"{split}: matches the published partition "
                f"(shape={serialized.shape}, maximum absolute difference={max_delta:.3g})"
            )
        else:
            print(f"{split}: no published CSV was present; generated shape={serialized.shape}")

        generated.to_csv(
            output_path,
            index=include_index,
            lineterminator="\r\n",
        )
        digest = hashlib.sha256(output_path.read_bytes()).hexdigest()
        results.append({
            "partition": split,
            "rows": len(generated),
            "spectral_variables": generated.shape[1] - 1,
            "sha256": digest,
        })

    return pd.DataFrame(results)


TASK_DIR = find_task_dir(TASK_FOLDER)
temporary_download = tempfile.TemporaryDirectory(prefix=f"nirbench_{TASK_FOLDER}_")
SOURCE_DIR = Path(temporary_download.name)
print(f"Preparing task {TASK_FOLDER}")

Preparing task 1-Wheat_kernels_protein


## Download and prepare

Download the authoritative source and apply the task specific conversion recorded in `DATA_LICENSES.md`.

In [3]:
archive_path = SOURCE_DIR / "wheat_kernels.zip"
download(SOURCE_URL, archive_path)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(SOURCE_DIR)

mat_path = next(SOURCE_DIR.rglob("NITSingleSeed.mat"))
source = loadmat(mat_path, squeeze_me=True)
wavelengths = [str(label).strip() for label in source["VarLabels_X"]]

train_data = pd.DataFrame(source["Calibration_X"], columns=wavelengths)
train_data = train_data.assign(protein=source["Calibration_Y"])

test_data = pd.DataFrame(source["Validation_X"], columns=wavelengths)
test_data = test_data.assign(protein=source["Validation_Y"])

assert train_data.shape == (415, 101)
assert test_data.shape == (108, 101)
print("Preserved source partitions: Calibration -> train; Validation -> test")

Downloaded wheat_kernels.zip (195,644 bytes)
Preserved source partitions: Calibration -> train; Validation -> test


## Verify and write

When published CSVs are present, compare their shapes, columns, row order, indices, and numerical values with the generated partitions. Then write `data_train.csv` and `data_test.csv` using deterministic CRLF line endings.

In [4]:
summary = verify_and_write(
    train_data,
    test_data,
    include_index=INCLUDE_INDEX,
    atol=COMPARISON_ATOL,
)
display(summary)
temporary_download.cleanup()
print("Temporary source files removed. Preparation complete.")

train: matches the published partition (shape=(415, 102), maximum absolute difference=0)
test: matches the published partition (shape=(108, 102), maximum absolute difference=0)


,partition,rows,spectral_variables,sha256
0,train,415,100,4742858ae617ffff7ce58098f65117898cf21afe96514a...
1,test,108,100,93b13c72113c71f938019be9ce7aeeb64b44f5e119eca2...


Temporary source files removed. Preparation complete.
